Current format check

In [2]:
from core.file_manager import preprocess_file_manager

from settings.main_settings import test_settings

import numpy as np

In [3]:
settings = test_settings().get_setting_dictionary()
preprocessed_steps = settings['preprocessed_steps']
preprocessing_steps_list = settings['preprocessing_steps_list']
channels = settings['channels']
original_data_folder = settings['original_data_folder']
target_spacing = settings['target_spacing']
crop_size = settings['crop_size']

file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [4]:
last_step = preprocessed_steps[preprocessing_steps_list[-1][0]]['end']
file_manager.current_load_step = last_step
patients = file_manager.get_file_names()

In [5]:
def get_patint_data_and_transform(patient):
    patient_data = file_manager.load_file_pickle(last_step,patient)
    patiend_data_shard = {
        'adc': patient_data['adc'],
        'dwi': patient_data['dwi'],
        't2': patient_data['t2'],
        'anatomy': patient_data['anatomy'],
        'lesion': patient_data['lesion'],

        'label': int(patient_data['lesion'].any())
    }
    return patiend_data_shard


Output that i need ... 

In [6]:
import os
import pickle
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage

def to_numpy(img):
    """Convert SimpleITK.Image -> numpy (x,y,z)."""
    if isinstance(img, sitk.Image):
        arr = sitk.GetArrayFromImage(img)      # (z,y,x)
        # arr = np.transpose(arr, (2, 1, 0))     # -> (x,y,z) unneeded 
        spacing = img.GetSpacing()
    else:
        arr = img
        #sus transpose 2, if i remove it form nifty to raw in preprocesing, then it will be great
        arr = np.transpose(arr, (2, 1, 0))
        spacing = None
    return arr, spacing

def prepare_dataset(patients, pp_dir):
    """
    dataset =
    {
        pid: {
            "adc": ndarray or sitk.Image,
            "dwi": ndarray or sitk.Image,
            "t2": ndarray or sitk.Image,
            "anatomy": ndarray or sitk.Image,
            "lesion": ndarray or sitk.Image,
            "label": int
        }
    }
    """

    os.makedirs(pp_dir, exist_ok=True)

    info_rows = []

    for patient in patients:

        pid, sample = (patient,get_patint_data_and_transform(patient))

        adc, spacing = to_numpy(sample["adc"])
        dwi, _ = to_numpy(sample["dwi"])
        t2, _ = to_numpy(sample["t2"])
        anatomy, _ = to_numpy(sample["anatomy"])
        lesion, _ = to_numpy(sample["lesion"])

        # ---------------------------
        # Build 4-channel image
        # ---------------------------

        img = np.stack([
            adc.astype(np.float32),
            dwi.astype(np.float32),
            t2.astype(np.float32),
            anatomy.astype(np.float32),   # prostate mask as 4th channel
        ], axis=-1)

        # ---------------------------
        # Lesion instance mask
        # ---------------------------

        lesion = lesion.astype(np.uint16)

        # If binary, convert to connected-component instances
        if np.unique(lesion).tolist() in ([0], [0, 1], [1]):
            lesion, _ = ndimage.label(lesion > 0)

        seg = lesion[..., None].astype(np.uint16)

        # ---------------------------
        # Save arrays
        # ---------------------------

        np.save(os.path.join(pp_dir, f"{pid}_img.npy"), img)
        np.save(os.path.join(pp_dir, f"{pid}_rois.npy"), seg)

        # ---------------------------
        # Metadata
        # ---------------------------

        fg_slices = np.where(lesion.sum(axis=(0, 1)) > 0)[0].tolist()

        meta = {
            "pid": pid,
            "class_target": [sample["label"]],  # MDT expects a list
            "spacing": spacing,
            "fg_slices": fg_slices,
        }

        with open(os.path.join(pp_dir, f"{pid}_meta_info.pickle"), "wb") as f:
            pickle.dump(meta, f)

        info_rows.append(meta)

    # Build info_df.pickle
    df = pd.DataFrame(info_rows)
    df.to_pickle(os.path.join(pp_dir, "info_df.pickle"))

    print(f"Prepared {len(info_rows)} patients.")

In [11]:
prepare_dataset(patients[1:3], '/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed/pp_dataset')

Prepared 2 patients.


In [8]:
img = np.load("/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed/pp_dataset/004_img.npy")
seg = np.load("/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed/pp_dataset/004_rois.npy")


# img = np.transpose(img, (2, 1, 0, 3)) 

print(img.shape)
print(seg.shape)

(24, 160, 160, 4)
(24, 160, 160, 1)


In [9]:
img = np.load("/home/robakp/Exeriments1/prostate_lesion_detection/preprocessed/merged/ProstateX-0003_img.npy")
seg = np.load("/home/robakp/Exeriments1/prostate_lesion_detection/preprocessed/merged/ProstateX-0003_rois.npy")

print(img.shape)
print(seg.shape)

(24, 160, 160, 8)
(24, 160, 160, 1)
